In [2]:
from pymongo import MongoClient
import requests

# 1. Połącz z MongoDB (serwer postawiony na Dockerze na porcie 27017)
client = MongoClient("mongodb://localhost:27017")
db = client.lab4
networks = db["networks"]

# Czyszczenie kolekcji przed ponownym wstawieniem danych
networks.delete_many({})

# 2. Pobierz dane z API
response = requests.get("https://api.geckoterminal.com/api/v2/networks")
data = response.json()["data"]

# 3. Wstaw dokumenty
networks.insert_many(data)
print(f"Pobrano i zapisano {len(data)} sieci do bazy MongoDB.")

# 4. Agregacja -- ile sieci per typ
# Uwaga: W odpowiedzi z live API pole typu znajduje się w głównym obiekcie jako 'type'
print("--- Agregacja według pola 'type' (rzeczywisty układ danych w API) ---")
pipeline_real = [
    {"$group": {"_id": "$type", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]
for doc in networks.aggregate(pipeline_real):
    print(doc)

# Wskazówka z zadania sugeruje agregację po 'attributes.type' (na wypadek innego formatu danych)
print("\n--- Agregacja według pola 'attributes.type' (podpowiedź z zadania) ---")
pipeline_hint = [
    {"$group": {"_id": "$attributes.type", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]
for doc in networks.aggregate(pipeline_hint):
    print(doc)

Pobrano i zapisano 100 sieci do bazy MongoDB.
--- Agregacja według pola 'type' (rzeczywisty układ danych w API) ---
{'_id': 'network', 'count': 100}

--- Agregacja według pola 'attributes.type' (podpowiedź z zadania) ---
{'_id': None, 'count': 100}
